try with https://huggingface.co/collections/dicta-il/dicta-lm-20-collection-661bbda397df671e4a430c27

In [1]:
# Import necessary libraries
import pandas as pd
from google.colab import drive, userdata
from openai import OpenAI
import time
import os


# --- 2. Mount Google Drive ---
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    print("Please ensure you have authorized Google Drive access.")
    # You might want to stop execution here if Drive is essential
    # exit()

Mounted at /content/drive
Google Drive mounted successfully.


In [2]:
#from getpass import getpass # For secure API key input if not using Colab secrets

# --- 3. LLM API Key Setup (OpenAI Example) ---
# Option 1: Using Colab Secrets (Recommended)
# 1. Click on the key icon (🔑) in the left sidebar.
# 2. Click "+ Add new secret".
# 3. Name: OPENAI_API_KEY
# 4. Value: your_openai_api_key_here
# 5. Toggle "Notebook access" to ON.
try:
    OPENAI_API_KEY = userdata.get('GrayzelOpenAI')
    if not OPENAI_API_KEY:
        raise ValueError("OpenAI API key not found in Colab secrets.")
    print("OpenAI API key loaded from Colab secrets.")
except Exception as e:
    print(f"Could not load API key from Colab secrets: {e}")
    print("Attempting to get API key via input.")
#    OPENAI_API_KEY = getpass('Enter your OpenAI API Key: ')
    if not OPENAI_API_KEY:
        print("No API key provided. Exiting.")
        exit()

client = OpenAI(api_key=OPENAI_API_KEY)

OpenAI API key loaded from Colab secrets.


In [4]:
# prompt: test open_api_key with a simple "what's up" question.

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": "What's up?"}
    ]
)

response.choices[0].message.content


'Not much, just here to help. How can I assist you today?'

In [5]:


# --- 1. Configuration & Setup ---

# Provided topics dictionary
TOPICS = {
    'Supernatural': [
        'Reincarnation', 'Perception', 'Light or fire as a sign of righteousness',
        'Prognostication', 'Pregnancy', 'Magic', 'Conversing with the dead',
        'Angels', 'Communicating with the invisible', 'Angel of death',
        'Demons', 'Miracle', 'Contraction of the road', 'Mystical vision'
    ],
    'Practice': [
        'Study', 'Healing', 'Protection', 'Solitude', 'Healing of the soul',
        'Drinking alcohol', 'Redemption of captives', 'Travel to other tsaddik',
        'Travel to the tsaddik', 'Devekut', 'Meditation', 'Tvilah',
        'Storytelling', 'Reception of hasidim', 'Sermon', 'Asceticism',
        'Fasting', 'Coping with alien thoughts', 'Torat ha tsaddik',
        'Dance', 'The travels of the tsaddik', 'Releasing agunot',
        'Pidyon monetary gift', 'Worship through corporeality',
        'Use of holy names', 'Writing amulets', 'Lifesaving', 'Smoking',
        'Music', 'Pilgrimage to the graves of tsaddikim', 'Pidyon nefesh',
        'Joke', 'Sexual abstinence', 'Failure', 'Business advice', 'Clothing'
    ],
    'Social relations': [
        'Inter hasidic master disciple relationship', 'With non hasidim',
        'Marriage', 'With the authorities', 'With non jews', 'With non jew'
    ],
    'Not a hasidic story': [
        'Hagiography', 'Sermon', 'Virtues', 'Halakhah'
    ],
    'Halakhah': [
        'Prayer', 'Circumcision', 'Kosher slaughtering', 'Tefillin', 'Tvilah'
    ],
    'Spaces': [
        'Heaven', 'The land of israel', 'Institutions beth midrash',
        'Institutions hassidic court', 'Mikveh', 'Hell', 'Rabbinical court'
    ],
    'Folkloristics': [
        'Story type advent', 'Story type rapprochement', 'Story type parable',
        'Story type virtues', 'Story type death of the tsaddik',
        'Story type protection', 'Story type birth of the tsaddik',
        'Story type inauguration', 'Story type revelation',
        'Story type prognostication', 'Story type joke'
    ],
    'Experience': [
        'Mystical', 'Journey to heaven', 'Enthusiasm', 'Devekut',
        'Dream', 'Revelation', 'Visions', 'Trembling'
    ],
    'Times': [
        'Holidays', 'Shabbat', 'Death', 'Historical event',
        'Inauguration', 'Birth', 'Eschatology'
    ],
    'Characters and roles': [
        'The family of the tsaddik', 'Hidden righteous', 'Tsaddik wife',
        'Agunah', 'Elijah the prophet', 'Preacher', 'Ritual slaughterer',
        'Baal shem', 'Magician', 'Messiah', # Tsaddik wife was duplicated, removed one
        'Sabbatai zevi', 'Witch'
    ],
    'Kabbalah': [
        'Kabbalistic terms yihudim', 'Kabbalistic terms kavvanot'
    ],
    'Knowledge': [
        'Esoteric', 'Practical kabbalah', 'Secret texts'
    ],
    'Custom': [
        'Hasidic custom', 'Clothing', 'Pilgrimage to the graves of tsaddikim'
    ]
}



In [13]:


# --- 4. Helper Functions ---

def format_topics_for_prompt(topics_dict):
    """Formats the topics dictionary into a string for the LLM prompt."""
    prompt_string = "Please classify the story into two of the following categories and subcategories.\n"
    prompt_string += "Respond ONLY with the categories in the format 'Topic-Category_Subtopic-Name'.\n"
    prompt_string += "For example: 'Characters-and-roles_Agunah' or 'Supernatural_Magic'.\n\n"
    prompt_string += "Available topics and subtopics:\n"
    for category, subtopics in topics_dict.items():
        # Sanitize category name for output format
        formatted_category = category.replace(" ", "-").replace("'", "")
        prompt_string += f"\n{category} (use '{formatted_category}' in output):\n"
        for subtopic in subtopics:
            formatted_subtopic = subtopic.replace(" ", "-").replace("'", "")
            prompt_string += f"  - {subtopic} (use '{formatted_subtopic}' in output, resulting in '{formatted_category}_{formatted_subtopic}')\n"
    prompt_string += "\nIf no category perfectly fits, choose the closest one. Do not make up new categories.\n"
    return prompt_string

TOPIC_PROMPT_TEXT = format_topics_for_prompt(TOPICS)

def get_story_classification(story_text, llm_client, model="gpt-3.5-turbo", max_retries=3, retry_delay=5):
    """
    Uses an LLM to classify a story based on the provided topics.
    """
    if not story_text or not isinstance(story_text, str) or len(story_text.strip()) < 10: # Basic check for empty/too short story
        return "Error_InvalidStoryText"

    system_prompt = "You are an expert academic assistant specializing in Jewish studies and folklore. Your task is to classify stories based on a predefined list of topics and subtopics."
    user_prompt = f"{TOPIC_PROMPT_TEXT}\n\nStory to classify:\n\"\"\"\n{story_text}\n\"\"\"\n\nClassification (Topic-Category_Subtopic-Name only):"

    for attempt in range(max_retries):
        try:
            response = llm_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2, # Lower temperature for more deterministic classification
                max_tokens=50 # Classification should be short
            )
            classification = response.choices[0].message.content.strip()

            # Basic validation of the output format
            if "_" in classification and not " " in classification:
                 # Further check if the returned topic and subtopic are valid
                parts = classification.split('_', 1)
                if len(parts) == 2:
                    main_topic_llm = parts[0].replace("-", " ") # Convert back for dict lookup
                    sub_topic_llm = parts[1].replace("-", " ")

                    # Find the original category name (case-insensitive match for flexibility)
                    original_main_topic = None
                    for t_key in TOPICS.keys():
                        if t_key.lower().replace(" ", "-") == parts[0].lower():
                            original_main_topic = t_key
                            break

                    if original_main_topic and any(s.lower().replace(" ", "-") == parts[1].lower() for s in TOPICS.get(original_main_topic, [])):
                        return classification # Valid format and seems to be from the list
                    else:
                        print(f"Warning: LLM returned a classification '{classification}' not strictly in the provided list for story: '{story_text[:50]}...'. Accepting it, but review needed.")
                        return classification # Accept if format is good, but content might be slightly off
                else: # Should not happen if "_" is present
                    print(f"Warning: LLM returned malformed classification '{classification}' for story: '{story_text[:50]}...'. Retrying...")

            else:
                print(f"Warning: LLM returned unexpected format: '{classification}' for story: '{story_text[:50]}...'. Retrying...")

            # If format is bad, it will fall through to the retry logic or return Error_LLM_BadFormat after retries

        except Exception as e:
            print(f"Error calling LLM API (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"Retrying in {retry_delay} seconds...")
                time.sleep(retry_delay)
            else:
                print("Max retries reached. Skipping this story.")
                return "Error_LLM_API_Failure"

    return "Error_LLM_BadFormat" # If all retries fail to get a good format


In [14]:
# prompt: try the prompt on a sample story:

sample_story = """
63.

שמעתי מר׳ יואל שהי׳ מ״מ בק"ק נעמרוב פ"א נסע הבעש״ט מן הדרך שמע כרוז א' משני דברים אילו אני מטיל עליך: או שיפלו עליך היידאמאקין ויגזלו ממך את ממונך והי׳ אצלו מעות סך שני מאות אדומים או שיהי' לך חולי קדחת והברירה השלישית שכחתי וחס על הממון מחמת שהי׳ ב"ח גדול בביתו ובירר לו חולי קדחת וכן הי׳ ובא עש"ק לק"ק שאריגראד ונחלש מאוד מחמת חולי' קדחת הנ"ל ואעפ״כ הלך לבה"מד ח והתפלל לפני התיבה ‧ והתפלה הי׳ להבת אש ואחר התפלה הוליכו אותו תחת ידיו מחמת חלישותו לאכסני׳ שלו וציוה שיביאו לו יין טוב מאוד ושתה כוס אחד והבריא ונתרפא מיד ואמר מילתא דבדיחות׳ טוב להיות הקדחת עוד שעה אחת ולשתות עוד כוס א׳ יין כי הוא טוב מאוד :
"""

classification = get_story_classification(sample_story, client)
print(f"Classification for the sample story: {classification}")


Classification for the sample story: Supernatural_Miracle


In [10]:

# --- 5. Load and Process TSV ---

# User Input for file paths and column name
tsv_file_path = '/content/drive/MyDrive/Hasidic_Tales_Final/data/raw_xml/stories_Sipurei-Kdoshim-Main.tsv' # MODIFY THIS

story_column_name = 'story' # MODIFY THIS if your column name is different
#story_column_name = input(f"Enter the name of the column containing the stories (default: '{default_story_column}'): ") or default_story_column

output_column_name = 'LLM_Classification'
output_tsv_path = tsv_file_path.replace('.tsv', '_classified.tsv')
#output_tsv_path = input(f"Enter the path to save the classified TSV file (default: {default_output_path}): ") or default_output_path

# Load the TSV file
try:
    print(f"Loading TSV file from: {tsv_file_path}")
    df = pd.read_csv(tsv_file_path, sep='\t', on_bad_lines='warn') # 'warn' or 'skip' for problematic lines
    print(f"Successfully loaded TSV. Shape: {df.shape}")
    print("Columns:", df.columns.tolist())
except FileNotFoundError:
    print(f"Error: File not found at {tsv_file_path}. Please check the path and try again.")
    exit()
except Exception as e:
    print(f"Error loading TSV file: {e}")
    exit()

if story_column_name not in df.columns:
    print(f"Error: Story column '{story_column_name}' not found in the TSV.")
    print(f"Available columns are: {df.columns.tolist()}")
    exit()

Loading TSV file from: /content/drive/MyDrive/Hasidic_Tales_Final/data/raw_xml/stories_Sipurei-Kdoshim-Main.tsv
Successfully loaded TSV. Shape: (17, 2)
Columns: ['id', 'story']


In [ ]:


# Prepare a list to store classifications
classifications = []

# --- Optional: Process a subset for testing ---
# num_rows_to_process = 5
# print(f"\n--- Processing the first {num_rows_to_process} stories for testing ---")
# df_subset = df.head(num_rows_to_process)
# Use df_subset instead of df in the loop below for testing

print(f"\n--- Starting classification of {len(df)} stories ---")
# Iterate through the DataFrame and classify each story
# Consider using df.itertuples() for slightly better performance on large DFs
for index, row in df.iterrows():
    story = row[story_column_name]
    print(f"\nProcessing story {index + 1}/{len(df)}...")
    if pd.isna(story) or not isinstance(story, str):
        print(f"Story {index + 1} is empty or not text. Skipping.")
        classification = "Error_EmptyOrInvalidStory"
    else:
        print(f"Story snippet: {story[:100]}...") # Print a snippet for context
        # Choose your model: "gpt-3.5-turbo", "gpt-4", "gpt-4-turbo", "gpt-4o" (newer, potentially better and cheaper than gpt-4-turbo)
        # gpt-3.5-turbo is cheaper and faster for testing.
        # gpt-4o or gpt-4-turbo are more powerful for final results.
        classification = get_story_classification(story, client, model="gpt-3.5-turbo") # CHANGE MODEL IF NEEDED
        print(f"Assigned classification: {classification}")

    classifications.append(classification)
    time.sleep(1) # Small delay to respect API rate limits if you have many rows and a free/low-tier plan.
                  # For paid plans with higher limits, you might reduce or remove this.

# Add the classifications as a new column
df[output_column_name] = classifications



--- Starting classification of 17 stories ---

Processing story 1/17...
Story snippet: מעשה מהרי"א הקדוש

אחותו של מרנא הר"י הקדוש ז"ל י היתה עקרה. אין לה ולד
					רבות בשנים והרבה פעמים ...
Error calling LLM API (attempt 1/3): Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 18592 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
Retrying in 5 seconds...
Error calling LLM API (attempt 2/3): Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 18592 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}
Retrying in 5 seconds...
Error calling LLM API (attempt 3/3): Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens.

In [ ]:

# --- 6. Save the Results ---
try:
    print(f"\nSaving classified data to: {output_tsv_path}")
    df.to_csv(output_tsv_path, sep='\t', index=False)
    print("Successfully saved the classified TSV file.")
    print("\n--- Sample of the output: ---")
    print(df[[story_column_name, output_column_name]].head())
except Exception as e:
    print(f"Error saving the output TSV file: {e}")

print("\n--- Classification Summary ---")
print(df[output_column_name].value_counts())

print("\n--- Script Finished ---")


Saving classified data to: /content/drive/MyDrive/Hasidic_Tales_Final/data/raw_xml/stories_Sipurei-Kdoshim-Main_classified.tsv
Successfully saved the classified TSV file.

--- Sample of the output: ---
                                               story  \
0  מעשה מהרי"א הקדוש\n\nאחותו של מרנא הר"י הקדוש ...   
1  מעשה מאחד שהשליך על ה׳ יהבו ובכל דרכיו הבטיח ב...   
2  מעשה מהרב הצדיק ר׳ ישראל בעש״ט אשר עדיין לא בא...   
3  מעשה מהרב הקדוש ר׳ זוסיא אחיו של הקדוש ר׳ אלימ...   
4  תולדות אפרים\n\t\t\t\t\tהוא מעשה נוראה מספר תו...   

                    LLM_Classification  
0                Error_LLM_API_Failure  
1                 Supernatural_Miracle  
2  Folkloristics_Story-type-protection  
3      Folkloristics_Story-type-advent  
4                Error_LLM_API_Failure  

--- Classification Summary ---
LLM_Classification
Folkloristics_Story-type-protection    7
Error_LLM_API_Failure                  2
Supernatural_Miracle                   2
Folkloristics_Story-type-advent        2